In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../asl_metadata.db")

In [8]:
import os

print(os.getcwd())
print(os.listdir())

/Users/sa17/Phase-3-Beyond-Words-Capstone/code
['eda.ipynb', 'sql_database.ipynb']


In [13]:
import os

print(os.listdir(".."))

['Background Research', 'code', 'docs', 'README.md', 'asl_metadata.db', '.gitignore', '.git', 'data']


In [14]:
import os

for item in os.listdir(".."):
    print(item)

Background Research
code
docs
README.md
asl_metadata.db
.gitignore
.git
data


In [16]:
import os

print(os.listdir("../data"))

[]


In [20]:
import os

train_dir = "/Users/sa17/Downloads/archive/asl_alphabet_train/asl_alphabet_train"

print(os.path.exists(train_dir))
print(os.listdir(train_dir)[:10])

True
['.DS_Store', 'R', 'U', 'I', 'N', 'G', 'Z', 'T', 'S', 'A']


In [21]:
for label in os.listdir(train_dir):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):
        print(label)

R
U
I
N
G
Z
T
S
A
F
O
H
del
nothing
space
M
J
C
D
V
Q
X
E
B
K
L
Y
P
W


In [22]:
import os

for label in sorted(os.listdir(train_dir)):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):

        count = len([
            f for f in os.listdir(class_path)
            if not f.startswith('.')
        ])

        print(f"{label}: {count}")

A: 3000
B: 3000
C: 3000
D: 3000
E: 3000
F: 3000
G: 3000
H: 3000
I: 3000
J: 3000
K: 3000
L: 3000
M: 3000
N: 3000
O: 3000
P: 3000
Q: 3000
R: 3000
S: 3000
T: 3000
U: 3000
V: 3000
W: 3000
X: 3000
Y: 3000
Z: 3000
del: 3000
nothing: 3000
space: 3000


In [23]:
records = []

for label in os.listdir(train_dir):

    if label.startswith('.'):
        continue

    class_path = os.path.join(train_dir, label)

    if os.path.isdir(class_path):

        for image_file in os.listdir(class_path):

            if image_file.startswith('.'):
                continue

            records.append({
                "file_path": os.path.join(class_path, image_file),
                "label": label
            })

df = pd.DataFrame(records)

df.head()

,file_path,label
0,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R
1,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R
2,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R
3,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R
4,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R


In [24]:
len(df)

87000

In [26]:

conn = sqlite3.connect("../asl_metadata.db")

df.to_sql(
    "images",
    conn,
    if_exists="replace",
    index=False
)

87000

In [28]:
pd.read_sql_query("PRAGMA table_info(images);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,file_path,TEXT,0,None,0
1,1,label,TEXT,0,None,0


In [29]:
df["is_valid"] = 1
df["split"] = "unassigned"

In [30]:
df.to_sql(
    "images",
    conn,
    if_exists="replace",
    index=False
)

87000

In [31]:
pd.read_sql_query("PRAGMA table_info(images);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,file_path,TEXT,0,None,0
1,1,label,TEXT,0,None,0
2,2,is_valid,INTEGER,0,None,0
3,3,split,TEXT,0,None,0


In [32]:
df = pd.read_sql_query("""
    SELECT file_path, label, split
    FROM images
    WHERE is_valid = 1
""", conn)

df.head()

,file_path,label,split
0,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R,unassigned
1,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R,unassigned
2,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R,unassigned
3,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R,unassigned
4,/Users/sa17/Downloads/archive/asl_alphabet_tra...,R,unassigned


In [33]:
from sklearn.model_selection import train_test_split

# First split: train vs temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# Second split: validation vs test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

df.loc[train_df.index, "split"] = "train"
df.loc[val_df.index, "split"] = "val"
df.loc[test_df.index, "split"] = "test"

In [35]:
df.to_sql("images", conn, if_exists="replace", index=False)

87000

In [36]:
pd.read_sql_query("""
SELECT split, COUNT(*) AS count
FROM images
GROUP BY split
""", conn)

,split,count
0,test,13050
1,train,60900
2,val,13050


In [37]:
pd.read_sql_query("""
SELECT label, split, COUNT(*) AS count
FROM images
GROUP BY label, split
ORDER BY label, split
""", conn)

,label,split,count
0,A,test,450
1,A,train,2100
2,A,val,450
3,B,test,450
4,B,train,2100
...,...,...,...
82,nothing,train,2100
83,nothing,val,450
84,space,test,450
85,space,train,2100
